# Predicting NFL Game Margins with Gaussian Processes
### Stat 345: Sports Analytics — Student Assignment

---

## Overview

In this assignment you will replicate the core methodology from Warner (2010), *"Winning isn't Everything: A Contextual Analysis of Results in the NFL Using Gaussian Processes."*

The paper uses a **Gaussian Process (GP)** model to predict NFL game margins of victory from team statistics, then compares the model's predictions to Vegas point spreads.

**By the end of this assignment you will be able to:**
- Explain what a Gaussian Process is and why it's useful for sports prediction
- Fit a GP model to real NFL data using `scikit-learn`
- Interpret both predictions *and* uncertainty estimates
- Evaluate your model against a Vegas benchmark
- Understand how kernel choice reflects modeling assumptions

**Estimated time:** 2–3 hours

**What to submit:** This completed notebook with all cells run and all written responses filled in.

---

## Before You Start: Read the Paper

Read Warner (2010), focusing on:
- **Section 2**: The features used (Figure 1) and the computed strength ranking (Section 2.2.1)
- **Section 3**: What a Gaussian Process is and the kernel choice
- **Section 4**: The results and comparison to Vegas

You do not need to understand every equation. The goal is to understand *what* the model does and *why* it was chosen.

---
## Background: What is a Gaussian Process?

Most regression models (like linear regression) give you a single predicted value for each input. A Gaussian Process goes one step further: **it gives you a full probability distribution over possible outcomes** — a predicted value *and* a measure of uncertainty.

Think of it this way. If you're asked to predict the margin of a game between two evenly matched teams with no recent history, you might say: "I think home team wins by 3, but I'm really not sure — could easily be anywhere from -14 to +20." That's a wide uncertainty. For a matchup between a dominant team and a weak one, you'd say "home wins by 14, pretty confident." That's a narrow uncertainty.

A GP captures exactly this: **predictions come with calibrated confidence intervals**, not just point estimates.

### The Kernel Function

The key ingredient of a GP is the **kernel** (also called the covariance function). It defines how similar two data points are to each other. The intuition: if two teams have very similar statistics, their game outcomes should be similar too.

The paper uses a **squared exponential (RBF) kernel**:

$$k(x_i, x_j) = \sigma^2 \exp\left(-\frac{\|x_i - x_j\|^2}{2\ell^2}\right)$$

- $\sigma^2$ controls overall variance (how spread out predictions are)
- $\ell$ is the **length scale**: how far apart two inputs need to be before their outputs are considered unrelated

A **WhiteKernel** is added to model pure noise — not everything about a football game is predictable from statistics.

### Why GP over Linear Regression for this problem?

Linear regression assumes the relationship between features and outcome is a straight line. The GP makes a weaker assumption: teams with *similar* statistics will have *similar* outcomes, but doesn't force that relationship to be linear. It also quantifies uncertainty, which turns out to be useful for building a betting strategy.

---
## Setup

Run the cell below to install and import all required packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# Data window — training seasons and test seasons
TRAIN_SEASONS = list(range(2018, 2023))  # 2018–2022
TEST_SEASONS  = [2023, 2024]
ALL_SEASONS   = TRAIN_SEASONS + TEST_SEASONS

print("Packages loaded.")
print(f"Training on: {TRAIN_SEASONS}")
print(f"Testing on:  {TEST_SEASONS}")

---
## Part 0: Load and Prepare the Data

The cells below build the dataset you'll use for the rest of the assignment. **You do not need to modify these — just run them.**

We use `nflverse`'s publicly hosted `games.csv`, which contains game results, scores, and Vegas spread lines for every NFL regular season game since 1999. From this we compute four features per team per game:

| Feature | Description |
|---|---|
| `win_pct` | Season-to-date winning percentage (prior games only) |
| `points_scored_avg` | Season-to-date average points scored per game |
| `points_allowed_avg` | Season-to-date average points allowed per game |
| `computed_strength` | Eigenvalue-based strength rating (Warner Section 2.2.1) |

Each feature is computed for both the **home team (H_)** and **away team (A_)**, giving 8 features per matchup.

> **Note:** All features use only games played *before* the current week. This prevents data leakage — the model only knows what was known at kick-off.

In [ ]:
# Load game results and spreads from nflverse
games = pd.read_csv(
    "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv",
    low_memory=False
)
games = games[
    (games["game_type"] == "REG") &
    (games["result"].notna()) &
    (games["season"].isin(ALL_SEASONS))
].copy()

# result = home_score - away_score (positive = home team won)
print(f"Games loaded: {len(games)} ({games['season'].min()}–{games['season'].max()})")

In [ ]:
# Build per-team per-game stats from game-level data
# Each game appears twice: once for home team, once for away team
home_rows = games[["game_id","season","week","home_team","away_team",
                   "home_score","away_score","result","spread_line"]].copy()
home_rows["team"]           = home_rows["home_team"]
home_rows["points_scored"]  = home_rows["home_score"]
home_rows["points_allowed"] = home_rows["away_score"]
home_rows["win"]            = (home_rows["home_score"] > home_rows["away_score"]).astype(int)

away_rows = games[["game_id","season","week","home_team","away_team",
                   "home_score","away_score","result","spread_line"]].copy()
away_rows["team"]           = away_rows["away_team"]
away_rows["points_scored"]  = away_rows["away_score"]
away_rows["points_allowed"] = away_rows["home_score"]
away_rows["win"]            = (away_rows["away_score"] > away_rows["home_score"]).astype(int)

team_games = pd.concat([home_rows, away_rows], ignore_index=True)
team_games = team_games.sort_values(["season","team","week"]).reset_index(drop=True)

# Season-to-date averages (prior games only, no leakage)
for col in ["points_scored", "points_allowed", "win"]:
    team_games[f"{col}_avg"] = (
        team_games.groupby(["season","team"])[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )
team_games = team_games.rename(columns={"win_avg": "win_pct"})

print(f"Team-game rows: {len(team_games):,}")

In [ ]:
# Compute eigenvalue-based team strength (Warner Section 2.2.1)
# See the paper for full derivation. In short: strength is a weighted average
# of your opponents' strengths, where the weight depends on how decisively you beat them.

def h_func(x):
    """Keener smoothing: h(x) = 0.5 + 0.5*sgn(x-0.5)*sqrt(|2x-1|)"""
    return 0.5 + 0.5 * np.sign(x - 0.5) * np.sqrt(np.abs(2 * x - 1))

def compute_strength(games_df):
    """Dominant eigenvector of the A matrix from prior games."""
    teams = sorted(set(games_df["home_team"]) | set(games_df["away_team"]))
    n = len(teams)
    idx = {t: i for i, t in enumerate(teams)}
    S = np.zeros((n, n))
    for _, row in games_df.iterrows():
        i, j = idx[row["home_team"]], idx[row["away_team"]]
        S[i][j] += row["home_score"]
        S[j][i] += row["away_score"]
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            total = S[i][j] + S[j][i]
            if total > 0:
                A[i][j] = h_func((S[i][j] + 1) / (total + 2))
    r = np.ones(n) / n
    for _ in range(200):
        r_new = A @ r
        norm = r_new.sum()
        if norm == 0: break
        r_new /= norm
        if np.allclose(r, r_new, atol=1e-10): break
        r = r_new
    return dict(zip(teams, r))

strength_records = []
for season in ALL_SEASONS:
    season_g = games[games["season"] == season].dropna(subset=["home_score","away_score"])
    for week in sorted(season_g["week"].unique()):
        prior = season_g[season_g["week"] < week]
        if len(prior) < 4: continue
        for team, s in compute_strength(prior).items():
            strength_records.append({"season": season, "week": week,
                                     "team": team, "computed_strength": s})

strength_df = pd.DataFrame(strength_records)
print(f"Strength records: {len(strength_df):,}")

In [ ]:
# Build the matchup-level dataset
# Each row = one game, with H_ (home) and A_ (away) features side by side

FEATURES = ["win_pct", "points_scored_avg", "points_allowed_avg"]

# Merge strength into team_games
team_games = team_games.merge(strength_df, on=["season","week","team"], how="left")
FEATURES += ["computed_strength"]

home_tg = team_games[team_games["team"] == team_games["home_team"]].copy()
away_tg = team_games[team_games["team"] == team_games["away_team"]].copy()

home_tg = home_tg.rename(columns={f: f"H_{f}" for f in FEATURES})
away_tg = away_tg.rename(columns={f: f"A_{f}" for f in FEATURES})

h_cols = [f"H_{f}" for f in FEATURES]
a_cols = [f"A_{f}" for f in FEATURES]

matchups = home_tg[["game_id","season","week","home_team","away_team"] + h_cols].merge(
    away_tg[["game_id"] + a_cols], on="game_id", how="inner"
).merge(
    games[["game_id","result","spread_line"]], on="game_id", how="left"
)

all_feat_cols = h_cols + a_cols
matchups = matchups.dropna(subset=all_feat_cols + ["result","spread_line"])
matchups = matchups.set_index("game_id").sort_values(["season","week"])

print(f"Matchup rows: {len(matchups):,}")
print(f"Features per game: {len(all_feat_cols)} ({len(h_cols)} home + {len(a_cols)} away)")
matchups[["season","week","home_team","away_team","result","spread_line"] + h_cols + a_cols].head()

---
## Part 1: Fit Your First Gaussian Process

Now the fun part. You'll fit a GP to predict `result` (home_score − away_score) from team statistics.

**A note on preprocessing:** GPs are sensitive to feature scale — a feature measured in hundreds (like yards per game) will dominate one measured in fractions (like win %). We standardize all features to zero mean and unit variance before fitting.

**Your task:** Complete the code below to:
1. Split matchups into train (2018–2022) and test (2023–2024)
2. Standardize features
3. Fit a `GaussianProcessRegressor` with `RBF() + WhiteKernel()`
4. Predict on the test set (use `return_std=True` to get uncertainty estimates)
5. Evaluate MAE and winner accuracy vs. the Vegas spread

In [ ]:
X_COLS = h_cols + a_cols   # the 8 features

# --- Split ---
train = matchups[matchups["season"].isin(TRAIN_SEASONS)].dropna(subset=X_COLS + ["result"])
test  = matchups[matchups["season"].isin(TEST_SEASONS)].dropna(subset=X_COLS + ["result","spread_line"])

# --- Standardize ---
scaler  = StandardScaler()
X_train = scaler.fit_transform(train[X_COLS])
X_test  = scaler.transform(test[X_COLS])
y_train = train["result"].values

# --- TODO: Define the GP kernel and fit the model ---
# Hint: use RBF() + WhiteKernel() and set normalize_y=True
kernel = # YOUR CODE HERE
gp     = # YOUR CODE HERE  (GaussianProcessRegressor)
# YOUR CODE HERE  (fit on X_train, y_train)

# --- TODO: Predict on the test set, retrieving both predictions and std ---
y_pred, y_std = # YOUR CODE HERE

test = test.copy()
test["gp_pred"] = y_pred
test["gp_std"]  = y_std

# --- Evaluate ---
mae_gp     = mean_absolute_error(test["result"], y_pred)
mae_spread = mean_absolute_error(test["result"], test["spread_line"])
acc_gp     = (np.sign(y_pred) == np.sign(test["result"])).mean()
acc_spread = (np.sign(test["spread_line"]) == np.sign(test["result"])).mean()

print(f"Test set: {len(test)} games ({TEST_SEASONS[0]}–{TEST_SEASONS[-1]})\n")
print(f"{'Metric':<28} {'GP Model':>10} {'Vegas':>10}")
print("-" * 50)
print(f"{'MAE (margin of victory)':<28} {mae_gp:>10.2f} {mae_spread:>10.2f}")
print(f"{'Winner accuracy':<28} {acc_gp:>9.1%} {acc_spread:>9.1%}")
print(f"\nFitted kernel: {gp.kernel_}")

### Part 1 — Written Questions

**Q1.** How does your GP model compare to Vegas on MAE and winner accuracy? What does this tell you about how well team statistics capture what Vegas knows?

*Your answer here:*

---

**Q2.** Look at the fitted kernel printed above (e.g. `RBF(length_scale=X.XX) + WhiteKernel(noise_level=X.XX)`). The `noise_level` parameter represents variance that the model *cannot* explain from the features. Is it large or small relative to the typical game margin? What does that tell you about NFL games?

*Your answer here:*

---
## Part 2: Visualizing Uncertainty

One of the GP's unique strengths is that every prediction comes with an uncertainty estimate (`gp_std`). Let's explore what that means in practice.

**Your task:** Create a scatter plot of predicted margin vs. actual margin, where each point is colored by the GP's uncertainty (`gp_std`). Then identify the games the model was most and least confident about.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Predicted vs Actual, colored by uncertainty ---
ax = axes[0]
sc = ax.scatter(test["result"], test["gp_pred"],
                c=test["gp_std"], cmap="RdYlGn_r", alpha=0.6, s=20)
ax.axline((0,0), slope=1, color="black", lw=1, linestyle="--", label="Perfect prediction")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
plt.colorbar(sc, ax=ax, label="GP uncertainty (std)")
ax.set_xlabel("Actual margin (home − away)")
ax.set_ylabel("GP predicted margin")
ax.set_title("Predicted vs Actual Margin\n(green = confident, red = uncertain)")
ax.legend()

# --- Plot 2: Distribution of uncertainty ---
ax2 = axes[1]
ax2.hist(test["gp_std"], bins=30, edgecolor="white", color="steelblue")
ax2.axvline(test["gp_std"].mean(), color="red", linestyle="--",
            label=f"Mean std = {test['gp_std'].mean():.2f}")
ax2.set_xlabel("GP uncertainty (std)")
ax2.set_ylabel("Number of games")
ax2.set_title("Distribution of GP Uncertainty")
ax2.legend()

plt.tight_layout()
plt.show()

# --- Most and least confident games ---
print("5 games the model was MOST confident about:")
cols = ["season","week","home_team","away_team","result","gp_pred","gp_std"]
print(test.nsmallest(5, "gp_std")[cols].to_string())

print("\n5 games the model was LEAST confident about:")
print(test.nlargest(5, "gp_std")[cols].to_string())

### Part 2 — Written Questions

**Q3.** Look at the 5 games the model was **least confident** about. Do these games have anything in common? (Think about the teams involved, the time of season, or anything unusual about the matchup.)

*Your answer here:*

---

**Q4.** The scatter plot shows predictions clustered in a narrower range than actual outcomes. Why does this happen? Is it a problem?

*Your answer here:*

---
## Part 3: Does Kernel Choice Matter?

The **Matérn kernel** is an alternative to RBF. The key difference:
- **RBF** assumes infinitely smooth functions — nearby inputs always produce very similar outputs
- **Matérn** allows for rougher functions — nearby inputs can have more abrupt differences

In sports, outcomes can change abruptly (injuries, weather, matchup quirks), so Matérn might be more realistic.

**Your task:** Refit the GP using `Matern() + WhiteKernel()` and compare performance to your RBF model.

In [ ]:
# --- TODO: Fit a GP with Matern kernel instead of RBF ---
kernel_matern = # YOUR CODE HERE
gp_matern     = # YOUR CODE HERE
# YOUR CODE HERE (fit)

y_pred_m, y_std_m = # YOUR CODE HERE (predict with return_std=True)

mae_m = mean_absolute_error(test["result"], y_pred_m)
acc_m = (np.sign(y_pred_m) == np.sign(test["result"])).mean()

print(f"{'Metric':<28} {'RBF kernel':>12} {'Matern kernel':>14} {'Vegas':>10}")
print("-" * 66)
print(f"{'MAE (margin of victory)':<28} {mae_gp:>12.2f} {mae_m:>14.2f} {mae_spread:>10.2f}")
print(f"{'Winner accuracy':<28} {acc_gp:>11.1%} {acc_m:>13.1%} {acc_spread:>9.1%}")
print(f"\nRBF kernel:    {gp.kernel_}")
print(f"Matern kernel: {gp_matern.kernel_}")

### Part 3 — Written Questions

**Q5.** Did switching to a Matérn kernel meaningfully change the results? Based on what you know about NFL games, which kernel's assumptions seem more appropriate and why?

*Your answer here:*

---

**Q6.** The kernel is a modeling *assumption* you bring to the data — it isn't learned from the data itself. Why does this matter? What could go wrong if your kernel assumptions don't match reality?

*Your answer here:*

---
## Part 4: Explore Your Own Feature Set

So far you've used all 8 features. Warner's original paper used only **4 features** selected via forward search. Let's explore whether feature choice matters.

**Your task:**
1. Try fitting the GP with only 2 features: `H_win_pct` and `A_win_pct` (the simplest possible model)
2. Then try adding `H_computed_strength` and `A_computed_strength`
3. Compare all three models (2 features, 4 features, all 8 features)

Does adding more features always help?

In [ ]:
results = []

feature_sets = {
    "Win % only (2 features)":         ["H_win_pct", "A_win_pct"],
    "Win % + Strength (4 features)":   ["H_win_pct", "A_win_pct",
                                         "H_computed_strength", "A_computed_strength"],
    "All features (8 features)":       X_COLS,
}

for label, feats in feature_sets.items():
    tr = train.dropna(subset=feats + ["result"])
    te = test.dropna(subset=feats + ["result"])

    # --- TODO: Standardize, fit GP (RBF + WhiteKernel), predict ---
    sc  = StandardScaler()
    Xtr = sc.fit_transform(tr[feats])
    Xte = sc.transform(te[feats])

    gp_tmp = # YOUR CODE HERE
    # YOUR CODE HERE (fit)
    preds  = # YOUR CODE HERE (predict, no std needed here)

    mae = mean_absolute_error(te["result"], preds)
    acc = (np.sign(preds) == np.sign(te["result"])).mean()
    results.append({"Model": label, "MAE": round(mae, 2), "Winner Acc": f"{acc:.1%}"})
    print(f"{label:<40}  MAE={mae:.2f}  Acc={acc:.1%}")

print(f"\n{'Vegas baseline':<40}  MAE={mae_spread:.2f}  Acc={acc_spread:.1%}")

### Part 4 — Written Questions

**Q7.** Does adding more features consistently improve model performance? If not, why might that be? (Think about what happens when you give a model more information to work with.)

*Your answer here:*

---

**Q8.** Across all your models, none come close to Vegas on winner accuracy. Based on your reading of the paper and your experiments here, what do you think Vegas knows that a GP trained on box-score statistics does not?

*Your answer here:*

---
## Part 5: The Betting Framework

The GP gives a predicted margin *and* an uncertainty estimate. Warner (Section 4.2) proposes using both to decide when to bet:

> Only bet when the model's prediction disagrees with Vegas by more than **k standard deviations**.

- **Bet home** when: `gp_pred − k × gp_std > spread_line`
- **Bet away** when: `gp_pred + k × gp_std < spread_line`
- **Home covers**: `result > spread_line`
- Break-even win rate with standard −110 vig: **52.4%**

**Your task:** Implement the betting rule and evaluate it across k values from 0.25 to 1.5.

In [ ]:
def evaluate_betting(test_df, k):
    # TODO: identify home bets and away bets using the rule above
    home_bets = # YOUR CODE HERE
    away_bets = # YOUR CODE HERE

    home_wins = (home_bets["result"] > home_bets["spread_line"]).sum()
    away_wins = (away_bets["result"] < away_bets["spread_line"]).sum()

    total_bets = len(home_bets) + len(away_bets)
    total_wins = int(home_wins + away_wins)
    win_rate   = total_wins / total_bets if total_bets > 0 else float("nan")
    return {"k": k, "bets": total_bets, "wins": total_wins, "win_rate": win_rate}

print(f"{'k':<6} {'Bets':>6} {'Wins':>6} {'Win Rate':>10}  {'Profitable?':>12}")
print("-" * 45)
for k in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]:
    r = evaluate_betting(test, k)
    flag = "YES" if r["win_rate"] > 0.524 else "no"
    print(f"{r['k']:<6} {r['bets']:>6} {r['wins']:>6} {r['win_rate']:>9.1%}  {flag:>12}")

print("\nBreak-even win rate (standard -110 vig): 52.4%")

### Part 5 — Written Questions

**Q9.** At low k values (many bets), is the strategy profitable? At high k values (few bets), what happens to the sample size? What does this tension tell you about the practical usefulness of this betting framework?

*Your answer here:*

---

**Q10.** Reflect on the overall project. Warner published this paper in 2010 using data from 2000–2009. We replicated his methodology on 2018–2024 data. Based on everything you've seen in this assignment, do you think a GP trained on box-score statistics would have been *more or less* effective against Vegas in 2010 than it is today? Why?

*Your answer here:*

---
## Submission Checklist

Before submitting, make sure:
- [ ] All code cells have been run and show output
- [ ] All 10 written questions have been answered
- [ ] Part 1: GP is fitted and results table is printed
- [ ] Part 2: Both plots are showing
- [ ] Part 3: Matérn comparison table is printed
- [ ] Part 4: All three feature set models have been evaluated
- [ ] Part 5: Betting framework table is printed

Upload the completed `.ipynb` file to Canvas.